# AgentEx Demo — Temporal Agent Notebook

Demonstrates using the AgentEx SDK to interact with the temporal agent:
1. Create a task
2. Ask the agent who it is
3. Ask what tools it has access to
4. Ask it to delegate to the LangChain agent

**Prerequisites:** The full stack must be running (`docker compose up -d` from repo root).

In [1]:
import os
import asyncio
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")  # loads root .env for SGP_* credentials

# SDK reads AGENTEX_BASE_URL to reach the backend
os.environ["AGENTEX_BASE_URL"] = "http://localhost:5003"

# Required by EnvironmentVariables.refresh() inside the SDK auth layer —
# not meaningful in a notebook context, but must be set to non-empty values.
os.environ.setdefault("AGENT_NAME", "notebook")
os.environ.setdefault("ACP_URL", "http://localhost")

from agentex.lib import adk
from agentex.types.text_content import TextContent

TEMPORAL_AGENT = "temporal-chat-agent-example"
POLL_INTERVAL = 2   # seconds between polls
POLL_TIMEOUT  = 60  # seconds before giving up

In [ ]:
async def send_and_wait(task_id: str, message: str) -> str:
    """Send an event to the temporal agent and poll until the agent reply arrives."""
    # Snapshot message count before sending so we can detect new ones
    before = await adk.messages.list(task_id=task_id)
    seen_ids = {m.id for m in before}

    print(f">>> {message}")
    await adk.acp.send_event(
        agent_name=TEMPORAL_AGENT,
        task_id=task_id,
        content=TextContent(author="user", content=message),
    )

    # Poll for a new text message from the agent (skip tool call/result messages)
    elapsed = 0
    while elapsed < POLL_TIMEOUT:
        await asyncio.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        messages = await adk.messages.list(task_id=task_id)
        new_agent_msgs = [
            m for m in messages
            if m.id not in seen_ids
            and getattr(m.content, "author", None) == "agent"
            and hasattr(m.content, "content")
        ]
        if new_agent_msgs:
            reply = new_agent_msgs[-1].content.content
            print(f"<<< {reply}")
            return reply

    raise TimeoutError(f"No agent reply after {POLL_TIMEOUT}s")

## 1. Create a Task

In [3]:
task = await adk.acp.create_task(agent_name=TEMPORAL_AGENT)
print(f"Created task: {task.id}")

2026-03-24 10:28:38,720 INFO [agentex.lib.environment_variables] [environment_variables.py:91] - Refreshing environment variables


Created task: 926c5008-3941-4d50-bda4-0e1c8f23057e


## 2. Ask who it is

In [4]:
await send_and_wait(task.id, "Who are you? Please introduce yourself.")

>>> Who are you? Please introduce yourself.
<<< Hello! I'm an AI assistant created by Anthropic. My role is to help you with a variety of tasks, from answering questions to providing analysis and creative assistance. I have access to a range of tools and capabilities that I can use to assist you.

Please feel free to ask me anything, and I'll do my best to help. I'm here to support you and work collaboratively to accomplish your goals. Let me know how I can be of assistance!


"Hello! I'm an AI assistant created by Anthropic. My role is to help you with a variety of tasks, from answering questions to providing analysis and creative assistance. I have access to a range of tools and capabilities that I can use to assist you.\n\nPlease feel free to ask me anything, and I'll do my best to help. I'm here to support you and work collaboratively to accomplish your goals. Let me know how I can be of assistance!"

## 3. Ask what tools it has

In [5]:
await send_and_wait(task.id, "What tools do you have access to?")

>>> What tools do you have access to?


AttributeError: 'ToolRequestContent' object has no attribute 'content'

## 4. Ask it to delegate to the LangChain agent

In [ ]:
await send_and_wait(task.id, "Please ask the LangChain agent what it does and report back what it says.")